# PMD-Net — Personalized Multi-timescale Deviation Network
### A research architecture motivated by the baseline-normalisation finding

**Status: research / post-deadline work.** This notebook implements a novel architecture
whose structure is derived from this project's central finding: stress is best inferred
from *how current physiology deviates from a personal expected baseline*, and this
deviation effect is largely invariant to the baseline's timescale.

**Design principles (each traceable to a finding):**
- **Learnable baseline-deviation layer** — instead of computing the Cosinor residual as a
  fixed preprocessing step, the network *learns* the expected baseline and subtracts it
  internally. (Motivated by U1: deviation features drive the gain.)
- **Multi-timescale baselines with attention** — several baseline estimators at different
  timescales; the model learns which to trust per subject/moment. (Motivated by ATBD: the
  timescale is flexible, so let the model weight it rather than fixing 24 h.)
- **Three information streams** — absolute state, deviation, and rate-of-change, fused by
  cross-attention. (Motivated by the physiological argument that identical absolute HRV
  means different things at different personal baselines.)
- **Ordinal (CORN) head** — respects the ordered stress levels. (Motivated by U3: severe
  ordinal errors are what matter.)

**Honesty note carried from ATBD.** Our timescale-invariance result predicts that the
adaptive-baseline mechanism may *match but not beat* the simple fixed-baseline model. This
notebook is therefore written as a proper controlled comparison: PMD-Net vs the existing
CNN-BiLSTM-Attention baseline, reporting the honest delta whichever way it falls. A null
result ("a principled architecture matches the baseline, confirming the mechanism is
already captured") is a legitimate and reportable outcome.


## 0. How this notebook is organised
1. Standard pipeline (WESAD preprocessing, windows) — reused from prior work.
2. Multi-timescale baseline estimators (the raw material the network consumes).
3. PMD-Net architecture, built in named blocks so each can be ablated.
4. The CORN ordinal head.
5. LOSO-CV training for both PMD-Net and the CNN-BiLSTM baseline.
6. Ablation harness (turn each novel block off, measure the drop).
7. Honest comparison + reporting guide.


## 1. Imports, Config, Pipeline (reused)

In [1]:
!pip install neurokit2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 22.2 MB/s eta 0:00:00


In [2]:
import os, json, pickle, warnings
import numpy as np, pandas as pd
from scipy.signal import welch
from scipy.optimize import curve_fit
from scipy.stats import wilcoxon
try:
    from scipy.integrate import trapezoid as TRAPZ
except ImportError:
    from scipy.integrate import trapz as TRAPZ
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import f1_score, cohen_kappa_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
import neurokit2 as nk
warnings.filterwarnings('ignore'); np.random.seed(42); tf.random.set_seed(42)
print("TF",tf.__version__,"GPU",len(tf.config.list_physical_devices('GPU'))>0)
DATA_PATH='/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD'
SAVE='/kaggle/working'; os.makedirs(SAVE,exist_ok=True)
SUBJECT_IDS=[2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
CLASS_NAMES=['relaxed','mild','moderate','high']; NCLS=4
WINDOW=120; STEP=5
print("Config loaded.")

TF 2.20.0 GPU True
Config loaded.


In [3]:
# --- standard helpers (same as prior notebooks) ---
def load_subject(sid):
    with open(f"{DATA_PATH}/S{sid}/S{sid}.pkl",'rb') as f: data=pickle.load(f,encoding='latin1')
    return data['signal']['chest'], data['signal']['wrist']['TEMP'].flatten(), data['label'].flatten()
def extract_rr(ecg,fs=700):
    ecg=nk.ecg_clean(ecg.flatten(),sampling_rate=fs); _,info=nk.ecg_peaks(ecg,sampling_rate=fs); rp=info['ECG_R_Peaks']
    return np.diff(rp)*(1000.0/fs),(rp[:-1]+rp[1:])/2.0/fs,rp
def clean_rr(rr,ts):
    rr=rr.copy().astype(float); rr[(rr<=300)|(rr>=2000)]=np.nan
    for i in range(1,len(rr)):
        if not np.isnan(rr[i-1]) and not np.isnan(rr[i]) and abs(rr[i]-rr[i-1])/rr[i-1]>0.20: rr[i]=np.nan
    m=np.isnan(rr)
    if m.any(): rr[m]=np.interp(np.where(m)[0],np.where(~m)[0],rr[~m])
    return rr,ts
def align_temp(wt,rp,fe=700,ft=4.0):
    tap=np.interp(rp/fe,np.arange(len(wt))/ft,wt); return (tap[:-1]+tap[1:])/2.0
def labels_to_rr(labels,rp):
    out=[]
    for i in range(len(rp)-1):
        seg=labels[rp[i]:rp[i+1]]; v=seg[seg>0]; out.append(0 if len(v)==0 else np.bincount(v).argmax())
    return np.array(out)
def roll_stat(rr,fn):
    o=np.zeros(len(rr))
    for i in range(len(rr)):
        w=rr[max(0,i-10):i+10]; o[i]=fn(w)
    return o
def roll_rmssd(rr): return roll_stat(rr, lambda w:(np.sqrt(np.mean(np.diff(w)**2)) if len(w)>1 else 0))
def roll_sdnn(rr):  return roll_stat(rr, lambda w:(np.std(w) if len(w)>1 else 0))

wesad={}
for sid in SUBJECT_IDS:
    try:
        chest,wt,labels=load_subject(sid); ecg=chest['ECG'].flatten()
        rr,ts,rp=extract_rr(ecg); temp=align_temp(wt,rp); rr,ts=clean_rr(rr,ts)
        rl=labels_to_rr(labels,rp); keep=rl>0
        rrk,tk,tsk,lk=rr[keep],temp[keep],ts[keep],rl[keep]
        new=np.zeros(len(lk),dtype=int); si=np.where(lk==2)[0]
        if len(si)>0:
            srr=rrk[si]; loc=[]
            for i in range(len(srr)):
                w=srr[max(0,i-15):i+15]; dd=np.diff(w); loc.append(np.sqrt(np.mean(dd**2)) if len(dd)>0 else 50)
            loc=np.array(loc); p33,p66=np.percentile(loc,33),np.percentile(loc,66)
            for i,idx in enumerate(si): new[idx]=(1 if loc[i]>=p66 else 2 if loc[i]>=p33 else 3)
        wesad[f'S{sid}']={'rr_ms':rrk,'temp':tk,'timestamps':tsk,'labels':new}
    except Exception as e: print("FAIL",sid,e)
print(len(wesad),"subjects preprocessed")

15 subjects preprocessed


## 2. Multi-Timescale Baseline Estimators
The novel input to PMD-Net. For each beat we precompute the *expected* HRV under several
timescales, so the network can attend over them. We use exponential moving averages (EMA)
at multiple half-lives as data-driven, causal baselines (causal = usable in real-time
deployment, unlike a full-session Cosinor fit).

Timescales (in beats, ~ half-lives):
- ultra-short ~30 beats  (~0.5 min)  — captures immediate drift
- short       ~120 beats (~2 min)
- medium      ~600 beats (~10 min)
- long        ~1800 beats (~30 min) — approaches session baseline

Each yields an expected-RR trajectory; the deviation is current minus expected.

In [4]:
def ema_baseline(x, halflife_beats):
    """Causal exponential moving average as a running baseline."""
    alpha = 1.0 - np.exp(np.log(0.5)/max(halflife_beats,1))
    out=np.zeros_like(x,dtype=float); out[0]=x[0]
    for i in range(1,len(x)):
        out[i]=alpha*x[i]+(1-alpha)*out[i-1]
    return out

BASELINE_HALFLIVES={'ultra':30,'short':120,'medium':600,'long':1800}

def build_baselines(rr):
    return {k:ema_baseline(rr,hl) for k,hl in BASELINE_HALFLIVES.items()}

# sanity demo on one subject
demo=wesad['S2']['rr_ms']
bl=build_baselines(demo)
print("baseline shapes:",{k:v.shape for k,v in bl.items()})
print("example deviations at beat 500:")
for k,v in bl.items():
    print(f"  {k:6s}: current={demo[500]:.1f}  expected={v[500]:.1f}  dev={demo[500]-v[500]:+.1f}")

baseline shapes: {'ultra': (3591,), 'short': (3591,), 'medium': (3591,), 'long': (3591,)}
example deviations at beat 500:
  ultra : current=937.1  expected=831.0  dev=+106.1
  short : current=937.1  expected=830.0  dev=+107.2
  medium: current=937.1  expected=863.6  dev=+73.5
  long  : current=937.1  expected=885.1  dev=+52.0


## 3. Window builder producing PMD-Net's three streams + multi-baselines

In [5]:
def build_pmd_windows(window=WINDOW, step=STEP):
    """Produce for each window:
       X_state : [RR, RMSSD, SDNN, HR] normalised          (absolute state)
       X_base  : expected RR at each timescale             (baselines)
       X_dev   : current - expected at each timescale       (deviation)
       X_rate  : first difference of RR (rate of change)    (dynamics)
       plus temp channel, labels, groups.
    """
    Xstate,Xbase,Xdev,Xrate,Xtemp,y,g=[],[],[],[],[],[],[]
    for sid,d in wesad.items():
        rr,temp,labels=d['rr_ms'],d['temp'],d['labels']
        bl=build_baselines(rr)
        rn=(rr-np.mean(rr))/(np.std(rr)+1e-8)
        rm=roll_rmssd(rn); sd=roll_sdnn(rn); hr=60000/(rr+1e-8)
        tn=(temp-np.mean(temp))/(np.std(temp)+1e-8)
        rate=np.concatenate([[0],np.diff(rr)])
        # normalise baselines/deviations per subject
        base_stack={k:(v-np.mean(rr))/(np.std(rr)+1e-8) for k,v in bl.items()}
        dev_stack ={k:(rr-v)/(np.std(rr)+1e-8) for k,v in bl.items()}
        raten=(rate-np.mean(rate))/(np.std(rate)+1e-8)
        for s in range(0,len(rr)-window,step):
            e=s+window; lab=labels[s+window//2]
            Xstate.append(np.stack([rn[s:e],rm[s:e],sd[s:e],hr[s:e],tn[s:e]],axis=-1))
            Xbase.append(np.stack([base_stack[k][s:e] for k in BASELINE_HALFLIVES],axis=-1))
            Xdev.append(np.stack([dev_stack[k][s:e] for k in BASELINE_HALFLIVES],axis=-1))
            Xrate.append(raten[s:e][:,None])
            y.append(lab); g.append(int(sid[1:]))
    return (np.array(Xstate,dtype=np.float32),np.array(Xbase,dtype=np.float32),
            np.array(Xdev,dtype=np.float32),np.array(Xrate,dtype=np.float32),
            np.array(y,dtype=np.int32),np.array(g,dtype=np.int32))

X_state,X_base,X_dev,X_rate,y_all,groups=build_pmd_windows()
print("state",X_state.shape,"base",X_base.shape,"dev",X_dev.shape,"rate",X_rate.shape)
print("classes",np.bincount(y_all))

state (11846, 120, 5) base (11846, 120, 4) dev (11846, 120, 4) rate (11846, 120, 1)
classes [8594 1101 1080 1071]


## 4. PMD-Net Architecture
Built from named blocks so the ablation harness (Section 7) can disable each novel
component and measure its contribution. The four inputs are the streams from Section 3.

```
 state (120x5)   base (120x4)   dev (120x4)   rate (120x1)
      |              |             |             |
 CNN-BiLSTM     BaselineAttn   (learnable       Conv1d
 encoder        over 4 scales   deviation       dynamics
      |          -> expected    refinement)      encoder
      |             baseline        |             |
      |              |             |             |
      +------ Cross-Attention fusion (state x deviation) ------+
                            |
                    concat[state, dev, rate, baseline-context]
                            |
                     Dense -> CORN ordinal head (3 logits)
```


In [6]:
class LearnableDeviation(layers.Layer):
    """Refines the raw current-minus-expected deviation with a learned gate,
    so the network can learn HOW MUCH of each timescale's deviation matters.
    Motivated by ATBD: let the model weight timescales instead of fixing one."""
    def __init__(self, n_scales, **kw):
        super().__init__(**kw); self.n_scales=n_scales
    def build(self,shp):
        self.scale_logits=self.add_weight(shape=(self.n_scales,),initializer='zeros',trainable=True,name='scale_w')
    def call(self, dev):           # dev: (B, T, n_scales)
        w=tf.nn.softmax(self.scale_logits)
        weighted=tf.reduce_sum(dev*w[None,None,:],axis=-1,keepdims=True)
        return weighted
    def get_scale_weights(self):
        return tf.nn.softmax(self.scale_logits).numpy()

# ---- INPUT ORDER CONTRACT (must match pmd_inputs() below) ----
#   state  -> always
#   dev    -> only if use_deviation
#   base   -> only if use_multiscale
#   rate   -> only if use_rate
def build_pmd_net(window=120, n_state=5, n_scales=4,
                  use_multiscale=True, use_deviation=True, use_rate=True, use_crossattn=True):
    state_in=tf.keras.Input((window,n_state),name='state')
    base_in =tf.keras.Input((window,n_scales),name='base')
    dev_in  =tf.keras.Input((window,n_scales),name='dev')
    rate_in =tf.keras.Input((window,1),name='rate')

    # --- state encoder (always on) ---
    x=layers.Conv1D(64,7,padding='same',activation='relu')(state_in)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(128,5,padding='same',activation='relu')(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Bidirectional(layers.LSTM(128,return_sequences=True))(x)
    x=layers.Dropout(0.4)(x)
    state_repr=layers.GlobalAveragePooling1D()(layers.Attention()([x,x]))

    parts=[state_repr]

    # --- deviation path (novel) ---
    if use_deviation:
        if use_multiscale:
            dev_weighted=LearnableDeviation(n_scales,name='learnable_dev')(dev_in)
        else:
            dev_weighted=layers.Lambda(lambda t:t[:,:,:1])(dev_in)
        dev_feat=layers.Conv1D(32,5,padding='same',activation='relu')(dev_weighted)
        dev_feat=layers.GlobalAveragePooling1D()(dev_feat)
        if use_crossattn:
            s_seq=layers.Reshape((1,-1))(state_repr)
            d_seq=layers.Reshape((1,-1))(layers.Dense(state_repr.shape[-1])(dev_feat))
            ca=layers.Attention()([s_seq,d_seq]); ca=layers.Flatten()(ca)
            parts.append(ca)
        parts.append(dev_feat)

    # --- baseline context (novel) ---
    if use_multiscale:
        b=layers.Conv1D(32,5,padding='same',activation='relu')(base_in)
        b=layers.GlobalAveragePooling1D()(b)
        parts.append(b)

    # --- rate-of-change dynamics (novel) ---
    if use_rate:
        r=layers.Conv1D(16,5,padding='same',activation='relu')(rate_in)
        r=layers.GlobalAveragePooling1D()(r)
        parts.append(r)

    fused=layers.Concatenate()(parts) if len(parts)>1 else parts[0]
    h=layers.Dense(64,activation='relu')(fused); h=layers.Dropout(0.4)(h)
    corn_logits=layers.Dense(NCLS-1,activation=None,name='corn')(h)

    # FIX: only declare inputs that are actually connected to the output,
    # otherwise Keras raises "inputs not connected to outputs" when a
    # branch is disabled during ablation. Order must match pmd_inputs().
    active_inputs=[state_in]
    if use_deviation:  active_inputs.append(dev_in)
    if use_multiscale: active_inputs.append(base_in)
    if use_rate:       active_inputs.append(rate_in)
    return Model(active_inputs, corn_logits, name='PMD_Net')

pmd=build_pmd_net()
print("PMD-Net params:",pmd.count_params())
print("input names:",[i.name for i in pmd.inputs])
pmd.summary()

I0000 00:00:1784693947.794912      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784693947.797986      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


PMD-Net params: 354887
input names: ['state', 'dev', 'base', 'rate']


Model: "PMD_Net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ state (InputLayer)  │ (None, 120, 5)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 120, 64)   │      2,304 │ state[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 120, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 60, 64)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 60, 128)   │     41,088 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 60, 128)   │        512 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 30, 128)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dev (InputLayer)    │ (None, 120, 4)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 30, 256)   │    263,168 │ max_pooling1d_1[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ learnable_dev       │ (None, 120, 1)    │          4 │ dev[0][0]         │
│ (LearnableDeviatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 30, 256)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 120, 32)   │        192 │ learnable_dev[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 30, 256)   │          0 │ dropout[0][0],    │
│ (Attention)         │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ conv1d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ attention[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │      8,448 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 256)    │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 256)    │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ base (InputLayer)   │ (None, 120, 4)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rate (InputLayer)   │ (None, 120, 1)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 354,887 (1.35 MB)

 Trainable params: 354,503 (1.35 MB)

 Non-trainable params: 384 (1.50 KB)

## 5. CORN Ordinal Head (validated encode/decode from the ordinal experiment)

In [7]:
def corn_labels(y,K=NCLS):
    y=np.asarray(y).reshape(-1,1); ks=np.arange(K-1).reshape(1,-1)
    return (y>ks).astype(np.float32)
def corn_loss(yb,logits):
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(yb,tf.sigmoid(logits)))
def corn_decode(logits):
    p=tf.sigmoid(logits).numpy(); pc=np.cumprod(p,axis=1)
    return (pc>0.5).sum(axis=1).astype(int)

def ordinal_metrics(yt,yp):
    yt,yp=np.asarray(yt),np.asarray(yp); err=np.abs(yp-yt)
    return dict(f1=f1_score(yt,yp,average='macro',zero_division=0),
                kappa=cohen_kappa_score(yt,yp,weights='quadratic'),
                mae=float(err.mean()),
                ma_mae=float(np.mean([err[yt==c].mean() for c in range(NCLS) if np.any(yt==c)])),
                dist=float((err>=2).mean()))
print("CORN utilities ready.")

CORN utilities ready.


## 6. Baseline Model for Comparison
The existing CNN-BiLSTM-Attention (softmax) on the same windows, so PMD-Net's value is
measured as a delta against the model you already have, not in a vacuum.

In [8]:
class SparseFocalLoss(tf.keras.losses.Loss):
    def __init__(self,gamma=2.0): super().__init__(); self.gamma=gamma
    def call(self,yt,yp):
        yt=tf.cast(yt,tf.int32); ce=tf.keras.losses.sparse_categorical_crossentropy(yt,yp)
        pt=tf.reduce_sum(tf.one_hot(yt,4)*yp,axis=-1); return tf.pow(1.0-pt,self.gamma)*ce
def build_baseline_cnn(window=120,n_state=5):
    si=tf.keras.Input((window,n_state))
    x=layers.Conv1D(64,7,padding='same',activation='relu')(si); x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(128,5,padding='same',activation='relu')(x); x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Bidirectional(layers.LSTM(128,return_sequences=True))(x); x=layers.Dropout(0.4)(x)
    x=layers.GlobalAveragePooling1D()(layers.Attention()([x,x]))
    x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.4)(x)
    return Model(si,layers.Dense(NCLS,activation='softmax')(x))
print("Baseline CNN ready.")

Baseline CNN ready.


## 7. LOSO-CV — PMD-Net vs Baseline

In [9]:
def class_weights(y):
    cw=compute_class_weight('balanced',classes=np.unique(y),y=y); return dict(enumerate(cw))

def pmd_inputs(idx, arch):
    """Return ONLY the input arrays the given architecture uses, in the same
    order build_pmd_net() declares them: state, [dev], [base], [rate]."""
    use_dev=arch.get('use_deviation',True)
    use_ms =arch.get('use_multiscale',True)
    use_rt =arch.get('use_rate',True)
    xs=[X_state[idx]]
    if use_dev: xs.append(X_dev[idx])
    if use_ms:  xs.append(X_base[idx])
    if use_rt:  xs.append(X_rate[idx])
    return xs

def train_pmd_fold(tr,te,return_model=False,**arch):
    ytr=corn_labels(y_all[tr])
    cw=class_weights(y_all[tr]); sw=np.array([cw[c] for c in y_all[tr]],dtype=np.float32)
    m=build_pmd_net(**arch)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss=lambda a,b:corn_loss(a,b))
    cb=[callbacks.EarlyStopping(monitor='val_loss',patience=12,restore_best_weights=True,verbose=0)]
    m.fit(pmd_inputs(tr,arch),ytr,sample_weight=sw,
          validation_split=0.15,epochs=100,batch_size=32,callbacks=cb,verbose=0)
    logits=m.predict(pmd_inputs(te,arch),verbose=0)
    pred=corn_decode(tf.convert_to_tensor(logits))
    return (pred,m) if return_model else pred

def train_base_fold(tr,te):
    m=build_baseline_cnn()
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss=SparseFocalLoss(2.0),metrics=['accuracy'])
    cb=[callbacks.EarlyStopping(monitor='val_loss',patience=12,restore_best_weights=True,verbose=0)]
    m.fit(X_state[tr],y_all[tr],validation_split=0.15,epochs=100,batch_size=32,
          class_weight=class_weights(y_all[tr]),callbacks=cb,verbose=0)
    return np.argmax(m.predict(X_state[te],verbose=0),axis=1)

logo=LeaveOneGroupOut()
yt_pmd,yp_pmd,yt_base,yp_base=[],[],[],[]
pmd_pf,base_pf={},{}
print("Running LOSO for PMD-Net and baseline (~60-90 min)...\n")
for tr,te in logo.split(X_state,y_all,groups):
    s=int(np.unique(groups[te])[0]); print(f"S{s:02d}",end=' ',flush=True)
    pp=train_pmd_fold(tr,te); pb=train_base_fold(tr,te)
    yt_pmd.extend(y_all[te]); yp_pmd.extend(pp)
    yt_base.extend(y_all[te]); yp_base.extend(pb)
    pmd_pf[s]=f1_score(y_all[te],pp,average='macro',zero_division=0)
    base_pf[s]=f1_score(y_all[te],pb,average='macro',zero_division=0)
    print(f"pmd={pmd_pf[s]:.3f} base={base_pf[s]:.3f}")
print("\nLOSO done.")

Running LOSO for PMD-Net and baseline (~60-90 min)...

S02 pmd=0.483 base=0.399
S03 pmd=0.612 base=0.670
S04 pmd=0.580 base=0.500
S05 pmd=0.447 base=0.316
S06 pmd=0.462 base=0.681
S07 pmd=0.553 base=0.545
S08 pmd=0.573 base=0.534
S09 pmd=0.301 base=0.391
S10 pmd=0.513 base=0.518
S11 pmd=0.602 base=0.383
S13 pmd=0.620 base=0.614
S14 pmd=0.567 base=0.368
S15 pmd=0.357 base=0.385
S16 pmd=0.578 base=0.562
S17 pmd=0.724 base=0.694

LOSO done.


## 8. Results — PMD-Net vs Baseline

In [10]:
mp=ordinal_metrics(yt_pmd,yp_pmd); mb=ordinal_metrics(yt_base,yp_base)
print("="*60); print("PMD-Net vs Baseline CNN-BiLSTM-Attention"); print("="*60)
print(f"{'metric':<20}{'baseline':>12}{'PMD-Net':>12}{'delta':>10}")
for k in ['f1','kappa','mae','ma_mae','dist']:
    delta=mp[k]-mb[k]; print(f"{k:<20}{mb[k]:>12.3f}{mp[k]:>12.3f}{delta:>+10.3f}")
print("="*60)
subs=sorted(pmd_pf.keys())
pf_p=np.array([pmd_pf[s] for s in subs]); pf_b=np.array([base_pf[s] for s in subs])
try:
    stat,p=wilcoxon(pf_p,pf_b); print(f"\nWilcoxon (F1) p={p:.4f}")
except Exception as e: print("Wilcoxon skipped",e)
d=(pf_p-pf_b).mean()/((pf_p-pf_b).std(ddof=1)+1e-12); print(f"Cohen's d (F1) {d:.3f}")
print("\nHONEST READ:")
print(" If PMD-Net BEATS baseline significantly -> novel architecture that works.")
print(" If PMD-Net MATCHES baseline (n.s.) -> confirms mechanism is already captured;")
print("   the principled structure is an interpretability gain, not an accuracy gain.")
print(" Either is reportable. Do NOT overclaim a non-significant delta.")

PMD-Net vs Baseline CNN-BiLSTM-Attention
metric                  baseline     PMD-Net     delta
f1                         0.534       0.566    +0.033
kappa                      0.750       0.774    +0.024
mae                        0.289       0.247    -0.042
ma_mae                     0.636       0.575    -0.061
dist                       0.072       0.052    -0.020

Wilcoxon (F1) p=0.2524
Cohen's d (F1) 0.249

HONEST READ:
 If PMD-Net BEATS baseline significantly -> novel architecture that works.
 If PMD-Net MATCHES baseline (n.s.) -> confirms mechanism is already captured;
   the principled structure is an interpretability gain, not an accuracy gain.
 Either is reportable. Do NOT overclaim a non-significant delta.


## 9. Ablation Harness
Turn off each novel block and measure the drop, isolating what (if anything) each
contributes. Runs on a single held-out fold for speed; expand to full LOSO for the paper.

In [11]:
def ablation_loso(arch, max_folds=None):
    """Run the ablation config across LOSO folds (default: all 15).
    Single-fold results are far too noisy to interpret, so we average
    across folds. Set max_folds=3 for a quick noisy preview."""
    yt,yp=[],[]
    for k,(tr,te) in enumerate(logo.split(X_state,y_all,groups)):
        if max_folds is not None and k>=max_folds: break
        pred=train_pmd_fold(tr,te,**arch)
        yt.extend(y_all[te]); yp.extend(pred)
    yt,yp=np.array(yt),np.array(yp)
    m=ordinal_metrics(yt,yp)
    return m['f1'], m['ma_mae']

configs={
 'full PMD-Net'        :dict(),
 'no multiscale'       :dict(use_multiscale=False),
 'no deviation path'   :dict(use_deviation=False),
 'no rate-of-change'   :dict(use_rate=False),
 'no cross-attention'  :dict(use_crossattn=False),
}

# NOTE: full LOSO for 5 configs is slow (~5x the main run).
# Set MAX_FOLDS=None for the reportable version; 3-5 for a quick scan.
MAX_FOLDS=None

print(f"Ablation (LOSO, max_folds={MAX_FOLDS}):")
abl_rows=[]
for name,cfg in configs.items():
    f1,mm=ablation_loso(cfg,max_folds=MAX_FOLDS)
    abl_rows.append((name,f1,mm))
    print(f"  {name:<20} F1={f1:.3f}  MA-MAE={mm:.3f}")

full_f1=abl_rows[0][1]; full_mm=abl_rows[0][2]
print("\nContribution of each block (drop when removed):")
for name,f1,mm in abl_rows[1:]:
    print(f"  {name:<20} dF1={f1-full_f1:+.3f}  dMA-MAE={mm-full_mm:+.3f}")
print("\nA block is CONTRIBUTING if removing it LOWERS F1 (negative dF1)")
print("or RAISES MA-MAE (positive dMA-MAE).")

Ablation (LOSO, max_folds=None):
  full PMD-Net         F1=0.545  MA-MAE=0.590
  no multiscale        F1=0.557  MA-MAE=0.572
  no deviation path    F1=0.564  MA-MAE=0.590
  no rate-of-change    F1=0.571  MA-MAE=0.566
  no cross-attention   F1=0.602  MA-MAE=0.512

Contribution of each block (drop when removed):
  no multiscale        dF1=+0.011  dMA-MAE=-0.018
  no deviation path    dF1=+0.019  dMA-MAE=-0.000
  no rate-of-change    dF1=+0.026  dMA-MAE=-0.023
  no cross-attention   dF1=+0.056  dMA-MAE=-0.078

A block is CONTRIBUTING if removing it LOWERS F1 (negative dF1)
or RAISES MA-MAE (positive dMA-MAE).


## 9b. Learned Timescale Weights (interpretability)
Even if PMD-Net does not significantly beat the baseline, the learned weighting over
baseline timescales is an interpretable result that connects directly to the
timescale-invariance finding.

In [12]:
# Interpretability payoff: which timescale does the model actually rely on?
# Train one full-config model on a single split and inspect the learned weights.
tr0,te0=next(logo.split(X_state,y_all,groups))
_,m_full=train_pmd_fold(tr0,te0,return_model=True)
try:
    ld=[l for l in m_full.layers if isinstance(l,LearnableDeviation)][0]
    w=ld.get_scale_weights()
    names=list(BASELINE_HALFLIVES.keys())
    print("Learned timescale weights (softmax over baselines):")
    for nm,wt,hl in zip(names,w,BASELINE_HALFLIVES.values()):
        print(f"  {nm:<7} (halflife {hl:>4} beats): {wt:.3f}")
    print("\nIf weight is spread roughly evenly, the model finds no single")
    print("timescale decisive -- consistent with the timescale-invariance finding.")
    print("If one dominates, that is an interpretable finding worth reporting.")
except IndexError:
    print("No LearnableDeviation layer found (multiscale disabled?).")

Learned timescale weights (softmax over baselines):
  ultra   (halflife   30 beats): 0.250
  short   (halflife  120 beats): 0.245
  medium  (halflife  600 beats): 0.255
  long    (halflife 1800 beats): 0.250

If weight is spread roughly evenly, the model finds no single
timescale decisive -- consistent with the timescale-invariance finding.
If one dominates, that is an interpretable finding worth reporting.


## 10. Reporting Guide (for the future paper)
Write whichever matches the result:

**If PMD-Net significantly beats the baseline:**
> "PMD-Net, whose structure explicitly models multi-timescale personal baseline deviation,
> improves macro-F1 / MA-MAE over a matched CNN-BiLSTM-Attention baseline (p < 0.05),
> demonstrating that building the deviation mechanism into the architecture is beneficial."

**If PMD-Net matches the baseline (likely, given ATBD):**
> "PMD-Net matches the baseline without significant improvement (p = X). This is consistent
> with our earlier finding that the baseline-deviation signal is already captured by simple
> residual features and is invariant to timescale: making the mechanism explicit yields
> interpretability (the learned scale weights are inspectable) rather than higher accuracy."

**Always report:** the learned timescale weights from `LearnableDeviation` (softmax over
scales) — even a null accuracy result gives an interpretable finding about which timescale
the model relies on, which ties back to ATBD.

**Do not:** claim architectural novelty as a performance win unless the Wilcoxon test
supports it. The honest, mechanism-consistent result is the stronger contribution.
